In [1]:
!pip -q install pypdf sentence-transformers scikit-learn google-generativeai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.5 MB/s eta 0:00:00


In [4]:
from pathlib import Path

dataset = Path("Dataset")

if not dataset.exists():
    raise FileNotFoundError(
        "Dataset folder not found. Place your PDF files inside a folder named 'Dataset'."
    )

pdf_files = sorted(dataset.glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF(s):\n")

for pdf in pdf_files:
    print(pdf.name)

Found 4 PDFs:

2607.26946v1.pdf
2607.27654v1.pdf
2607.28464v1.pdf
2607.28496v1.pdf


In [5]:
from pypdf import PdfReader

documents = []

for pdf in pdf_files:

    reader = PdfReader(pdf)

    text = ""

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    documents.append({
        "file": pdf.name,
        "text": text
    })

print(f"Loaded {len(documents)} documents.")

Loaded 4 documents.


In [6]:
print(documents[0]["file"])

print()

print(documents[0]["text"][:1000])

2607.26946v1.pdf

Belief-Guided Decision Making with Uncertainty 
Gating in the Game of Go 
 
1st Mehrad Yaghoubi 
Department of Computer Engineering, Ka.C, Islamic 
Azad University, Karaj, Iran 
me.yaghoubi@iau.ir 
3rd Abbas Jalilvand 
Department of Computer Engineering, Ka.C, Islamic 
Azad University, Karaj, Iran 
jalilvand@iau.ac.ir 
2nd Azam Bastanfard* 
Department of Computer Engineering, Ka.C, Islamic 
Azad University, Karaj, Iran 
bastanfard@iau.ac.ir 
4th Ashkan Rezaei 
Department of Computer Engineering, Ka.C, Islamic 
Azad University, Karaj, Iran 
ashkan.rezaei0702@iau.ir 
 
 
Abstract—Recent advancements in Computer Go, driven by 
AlphaZero and MuZero, rely heavily on Monte Carlo Tree 
Search (MCTS) to correct the errors of the neural network 
policy. While effective on massive computational clusters, this 
dependence creates a critical bottleneck on consumer -grade 
hardware (e.g., R TX 2060), where the computational cost of  
tree management severely limits inference rates

In [7]:
def chunk_text(text, chunk_size=500, overlap=100):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):

        end = min(start + chunk_size, len(words))

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        if end == len(words):
            break

        start = end - overlap

    return chunks

In [8]:
chunked_documents = []

for doc in documents:

    chunks = chunk_text(doc["text"])

    for i, chunk in enumerate(chunks):

        chunked_documents.append({
            "file": doc["file"],
            "chunk_id": i,
            "text": chunk
        })

print(f"Total Chunks: {len(chunked_documents)}")

Total Chunks: 59


In [9]:
print(chunked_documents[0]["file"])
print(chunked_documents[0]["chunk_id"])

print()

print(chunked_documents[0]["text"][:700])

2607.26946v1.pdf
0

Belief-Guided Decision Making with Uncertainty Gating in the Game of Go 1st Mehrad Yaghoubi Department of Computer Engineering, Ka.C, Islamic Azad University, Karaj, Iran me.yaghoubi@iau.ir 3rd Abbas Jalilvand Department of Computer Engineering, Ka.C, Islamic Azad University, Karaj, Iran jalilvand@iau.ac.ir 2nd Azam Bastanfard* Department of Computer Engineering, Ka.C, Islamic Azad University, Karaj, Iran bastanfard@iau.ac.ir 4th Ashkan Rezaei Department of Computer Engineering, Ka.C, Islamic Azad University, Karaj, Iran ashkan.rezaei0702@iau.ir Abstract—Recent advancements in Computer Go, driven by AlphaZero and MuZero, rely heavily on Monte Carlo Tree Search (MCTS) to correct the errors of


In [10]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


In [11]:
texts = [doc["text"] for doc in chunked_documents]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding matrix shape:", embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding matrix shape: (59, 384)


In [12]:
print("First embedding vector (first 10 values):")
print(embeddings[0][:10])

First embedding vector (first 10 values):
[-0.03851485 -0.00529998  0.01295824  0.04762311  0.05482321 -0.11501299
 -0.03028288  0.04581652  0.04816965  0.02120905]


In [13]:
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
def retrieve(query, top_k=3):

    # Convert the query into an embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Compare query with every document chunk
    similarities = cosine_similarity(query_embedding, embeddings)[0]

    # Get indices of the most similar chunks
    top_indices = similarities.argsort()[-top_k:][::-1]

    results = []

    for idx in top_indices:
        results.append({
            "file": chunked_documents[idx]["file"],
            "chunk_id": chunked_documents[idx]["chunk_id"],
            "score": similarities[idx],
            "text": chunked_documents[idx]["text"]
        })

    return results

In [15]:
results = retrieve("What is a Vision Transformer?")

for i, result in enumerate(results, start=1):

    print("=" * 80)
    print(f"Result {i}")
    print(f"File      : {result['file']}")
    print(f"Chunk     : {result['chunk_id']}")
    print(f"Similarity: {result['score']:.4f}\n")
    print(result["text"][:500])
    print()

Result 1
File      : 2607.26946v1.pdf
Chunk     : 3
Similarity: 0.4270

a Vision Transformer backbone. A. Architecture and Backbone The core architecture processes information through five distinct phases, transitioning from raw spatial inputs to high - level strategic belief and policy decisions. 1) Input and Spatial Embedding: The data flow begins with a game state observation containing 4 channels (Current player stones, Opponent stones, Empty spots, History). This input is projected into an embedding space via a 1 × 1 convolutional layer. nosep • Tokenization: T

Result 2
File      : 2607.28464v1.pdf
Chunk     : 14
Similarity: 0.3138

the goal of this experiment is to analyze the robust- ness of the vision-language model (VLM), we exclude the segmentation model from this analysis to ensure that only the VLM’s performance is evaluated. Consequently, eval- uation is based solely on the bounding-box IoU predicted by the VLM during inference. For this reason, the results reported in th

In [16]:
retrieve("What are Large Language Models?")

[{'file': '2607.27654v1.pdf',
  'chunk_id': 18,
  'score': np.float32(0.626417),
  'text': 'Analysis of Large Language Models SIGIR ’26, July 20–24, 2026, Melbourne, VIC, Australia. [46] Zairun Yang, Yilin Wang, Zhengyan Shi, Yuan Yao, Lei Liang, Keyan Ding, Emine Yilmaz, Huajun Chen, and Qiang Zhang. 2025. EventRAG: Enhancing LLM Generation with Event Knowledge Graphs. InProceedings of the 63rd Annual Meeting of the Association for Computational Linguistics (Volume 1: Long Papers). 16967–16979. [47] Zhihan Zhang, Yixin Cao, Chenchen Ye, Yunshan Ma, Lizi Liao, and Tat-Seng Chua. 2024. Analyzing Temporal Complex Events with Large Language Models? A Benchmark towards Temporal, Long Context Understanding. InProceedings of the 62nd Annual Meeting of the Association for Computational Linguistics (Volume 1: Long Papers). 1588–1606.'},
 {'file': '2607.27654v1.pdf',
  'chunk_id': 16,
  'score': np.float32(0.4459488),
  'text': 'Continuous treatment effect modeling in multi-agent dynamical syst

In [17]:
retrieve("Explain transformer architecture")

[{'file': '2607.26946v1.pdf',
  'chunk_id': 3,
  'score': np.float32(0.36578494),
  'text': 'a Vision Transformer backbone. A. Architecture and Backbone The core architecture processes information through five distinct phases, transitioning from raw spatial inputs to high - level strategic belief and policy decisions. 1) Input and Spatial Embedding: The data flow begins with a game state observation containing 4 channels (Current player stones, Opponent stones, Empty spots, History). This input is projected into an embedding space via a 1 × 1 convolutional layer. nosep • Tokenization: The 19 × 19 board is flattened into 361 tokens. Following the Vision Transformer approach in board games [33], this method outperforms classical CNNs in capturing global board structures. • Positional Encoding: We utilize a Fixed2DPositionalEncoding to inject geometric information, enabling the model to understand Manhattan distances and stone adjacency. As noted by Shi et al. [32], this encoding is vital

In [18]:
retrieve("What is computer vision?")

[{'file': '2607.26946v1.pdf',
  'chunk_id': 3,
  'score': np.float32(0.35632858),
  'text': 'a Vision Transformer backbone. A. Architecture and Backbone The core architecture processes information through five distinct phases, transitioning from raw spatial inputs to high - level strategic belief and policy decisions. 1) Input and Spatial Embedding: The data flow begins with a game state observation containing 4 channels (Current player stones, Opponent stones, Empty spots, History). This input is projected into an embedding space via a 1 × 1 convolutional layer. nosep • Tokenization: The 19 × 19 board is flattened into 361 tokens. Following the Vision Transformer approach in board games [33], this method outperforms classical CNNs in capturing global board structures. • Positional Encoding: We utilize a Fixed2DPositionalEncoding to inject geometric information, enabling the model to understand Manhattan distances and stone adjacency. As noted by Shi et al. [32], this encoding is vital

In [19]:
retrieve("How does attention work?")

[{'file': '2607.28464v1.pdf',
  'chunk_id': 13,
  'score': np.float32(0.28432733),
  'text': 'AAAI Conference on Artificial Intelligence, pages 11022–11030, 2025. 6, 7 10 Supplementary Material Can Vision-Language Models Reason about AI Edits in Images? A. Reasoning Trace Analysis Without any supervision on the reasoning output, our model learns to leverage the model’s inherent reasoning capabil- ities to make the decision on labeling and localizing the edited region. Some examples from each dataset clearly showcase this capability, as in Figures 7, 8, 9, 10. B. Ablation study on True-Negative Reward As we mentioned in the main paper, the reward given for correctly predicting an authentic image as an untampered image was a hyperparameter. It had to be carefully tuned. Otherwise, the model would learn to hack the rewards by always predicting them as untampered to score on only the untampered data. Tab. 4 studies the true-negative reward weightwusing only the VLM bounding-box output, bef

In [20]:
def answer_question(question):

    results = retrieve(question)

    print(f"Question: {question}\n")

    print("Most Relevant Context:\n")

    for i, result in enumerate(results, 1):
        print(f"Result {i}")
        print(f"Paper: {result['file']}")
        print(f"Similarity Score: {result['score']:.3f}")
        print(result['text'][:500])
        print("-"*80)

In [22]:
answer_question("What are Large Language Models?")

answer_question("What is a Vision Transformer?")

answer_question("How does attention work?")

answer_question("What is computer vision?")

answer_question("What are the contributions of these papers?")

Question: What are Large Language Models?

Most Relevant Context:

Result 1
Paper: 2607.27654v1.pdf
Similarity Score: 0.626
Analysis of Large Language Models SIGIR ’26, July 20–24, 2026, Melbourne, VIC, Australia. [46] Zairun Yang, Yilin Wang, Zhengyan Shi, Yuan Yao, Lei Liang, Keyan Ding, Emine Yilmaz, Huajun Chen, and Qiang Zhang. 2025. EventRAG: Enhancing LLM Generation with Event Knowledge Graphs. InProceedings of the 63rd Annual Meeting of the Association for Computational Linguistics (Volume 1: Long Papers). 16967–16979. [47] Zhihan Zhang, Yixin Cao, Chenchen Ye, Yunshan Ma, Lizi Liao, and Tat-Seng Chua. 2024. 
--------------------------------------------------------------------------------
Result 2
Paper: 2607.27654v1.pdf
Similarity Score: 0.446
Continuous treatment effect modeling in multi-agent dynamical systems. In Proceedings of the ACM Web Conference 2024. 4607–4617. [22] Kimi Team. 2025. Kimi K2: Open Agentic Intelligence.CoRRabs/2507.20534 (2025). [23] Bobo Li, Xudong Han